# Hyperparameter Search — BiLSTM+Attention (Optuna)

Mencari kombinasi hyperparameter terbaik berdasarkan **val_loss** (minimize) dan **val_accuracy** (maximize) menggunakan Optuna dengan pruning.

**Search space:**
| Parameter | Range |
|---|---|
| `lr` | 1e-4 .. 1e-2 (log) |
| `dropout` | 0.2 .. 0.6 |
| `weight_decay` | 1e-5 .. 1e-2 (log) |
| `label_smoothing` | 0.0 .. 0.2 |
| `embed_dim` | 64, 128 |
| `hidden_dim1` | 128, 256 |
| `hidden_dim2` | 64, 128 |
| `batch_size` | 32, 64, 128 |

**Output:** `../models/model_dl/optuna_best_params.json`


## Install & Environment


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna', '-q'])

import os
IS_KAGGLE = os.path.exists('/kaggle')
DATA_PATH  = '/kaggle/input/sentiment-cleaned/cleaned_text.csv' if IS_KAGGLE else '../data/processed/cleaned_text.csv'
OUTPUT_DIR = '/kaggle/working/' if IS_KAGGLE else '../models/model_dl/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Environment: {"Kaggle" if IS_KAGGLE else "Lokal"}')
print(f'Data       : {DATA_PATH}')

## 1. Import


In [ ]:
import json, logging, time, warnings
from collections import Counter
from dataclasses import dataclass

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import optuna
from optuna.trial import TrialState
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## 2. Konfigurasi Search


In [ ]:
# --- Konfigurasi search ---
N_TRIALS       = 30     # jumlah kombinasi yang dicoba
EPOCHS_PER_TRIAL = 5   # epoch per trial
VOCAB_SIZE     = 30_000
MAX_LEN        = 128
TEST_SIZE      = 0.2
SEED           = 42

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
print(f'N_TRIALS: {N_TRIALS} | Epochs/trial: {EPOCHS_PER_TRIAL}')

## 3. Definisi Class


In [ ]:
class Vocabulary:
    PAD_IDX, UNK_IDX = 0, 1
    def __init__(self, max_size=30_000):
        self.max_size = max_size
        self.word2idx = {'<PAD>': 0, '<UNK>': 1}
        self.idx2word = {0: '<PAD>', 1: '<UNK>'}
    def build(self, texts):
        counter = Counter()
        for t in texts: counter.update(str(t).split())
        for word, _ in counter.most_common(self.max_size - 2):
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
        return self
    def encode(self, text, max_len=128):
        tokens = str(text).split()[:max_len]
        ids = [self.word2idx.get(t, self.UNK_IDX) for t in tokens]
        ids += [self.PAD_IDX] * (max_len - len(ids))
        return ids
    def __len__(self): return len(self.word2idx)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=128):
        self.X = torch.tensor([vocab.encode(t, max_len) for t in texts], dtype=torch.long)
        self.y = torch.tensor(labels, dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.y[i]

class BiLSTMAttention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim1, hidden_dim2, num_classes=2, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm1 = nn.LSTM(embed_dim,     hidden_dim1, batch_first=True, bidirectional=True)
        self.lstm2 = nn.LSTM(hidden_dim1*2, hidden_dim2, batch_first=True, bidirectional=True)
        self.attention = nn.Linear(hidden_dim2*2, 1)
        self.dropout   = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim2*2, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        emb    = self.dropout(self.embedding(x))
        out1,_ = self.lstm1(emb);  out1 = self.dropout(out1)
        out2,_ = self.lstm2(out1); out2 = self.dropout(out2)
        attn_w = torch.softmax(self.attention(out2), dim=1)
        ctx    = (attn_w * out2).sum(dim=1)
        return self.fc(self.dropout(ctx))

print('✅ Class definitions OK')

## 4. Fungsi Train & Evaluate


In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(X)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss/n, correct/n

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        out = model(X)
        total_loss += criterion(out, y).item() * len(y)
        correct += (out.argmax(1) == y).sum().item()
        n += len(y)
    return total_loss/n, correct/n

print('✅ Fungsi train/eval OK')

## 5. Load Data (sekali, dibagi ulang tiap trial)


In [ ]:
df = pd.read_csv(DATA_PATH)
df['clean_text'] = df['clean_text'].fillna('').astype(str)
all_texts  = df['clean_text'].tolist()
all_labels = df['sentiment'].tolist()

# Build vocabulary satu kali
vocab = Vocabulary(max_size=VOCAB_SIZE).build(all_texts)

X_train, X_val, y_train, y_val = train_test_split(
    all_texts, all_labels, test_size=TEST_SIZE, random_state=SEED, stratify=all_labels
)
print(f'✅ Data: {len(df)} sampel | Train: {len(X_train)} | Val: {len(X_val)}')
print(f'   Vocab: {len(vocab)} token')

## 6. Objective Function (Optuna)


In [ ]:
def objective(trial):
    # --- Search space ---
    lr               = trial.suggest_float('lr',               1e-4, 1e-2, log=True)
    dropout          = trial.suggest_float('dropout',          0.2,  0.6)
    weight_decay     = trial.suggest_float('weight_decay',     1e-5, 1e-2, log=True)
    label_smoothing  = trial.suggest_float('label_smoothing',  0.0,  0.2)
    embed_dim        = trial.suggest_categorical('embed_dim',        [64, 128])
    hidden_dim1      = trial.suggest_categorical('hidden_dim1',      [128, 256])
    hidden_dim2      = trial.suggest_categorical('hidden_dim2',      [64, 128])
    batch_size       = trial.suggest_categorical('batch_size',       [32, 64, 128])

    # --- DataLoader ---
    train_ds = SentimentDataset(X_train, y_train, vocab, MAX_LEN)
    val_ds   = SentimentDataset(X_val,   y_val,   vocab, MAX_LEN)
    train_loader = DataLoader(train_ds, batch_size=batch_size,   shuffle=True,  num_workers=0, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size*2, shuffle=False, num_workers=0, pin_memory=True)

    # --- Model ---
    set_seed(SEED)
    model = BiLSTMAttention(
        vocab_size  = len(vocab),
        embed_dim   = embed_dim,
        hidden_dim1 = hidden_dim1,
        hidden_dim2 = hidden_dim2,
        dropout     = dropout
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=1)

    best_val_loss = float('inf')
    best_val_acc  = 0.0

    for epoch in range(1, EPOCHS_PER_TRIAL + 1):
        train_epoch(model, train_loader, criterion, optimizer)
        vl_loss, vl_acc = eval_epoch(model, val_loader, criterion)
        scheduler.step(vl_loss)

        # Intermediate value untuk pruning
        trial.report(vl_loss, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_val_acc  = vl_acc

    # Simpan val_acc sebagai user attr agar bisa diambil nanti
    trial.set_user_attr('val_acc', round(best_val_acc, 6))
    return best_val_loss   # minimize

print('✅ Objective function siap')

## 7. Jalankan Optuna Study


In [ ]:
sampler = optuna.samplers.TPESampler(seed=SEED)          # Bayesian (Tree Parzen Estimator)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)

study = optuna.create_study(
    direction='minimize',        # minimize val_loss
    sampler=sampler,
    pruner=pruner,
    study_name='bilstm_hpo'
)

print(f'🔍 Mulai search: {N_TRIALS} trials × {EPOCHS_PER_TRIAL} epoch...')
t0 = time.time()
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
elapsed = time.time() - t0
print(f'\n⏱ Selesai dalam {elapsed/60:.1f} menit')

## 8. Tampilkan Hasil


In [ ]:
import optuna.visualization.matplotlib as oviz
import matplotlib.pyplot as plt

best = study.best_trial
print('=' * 60)
print('  BEST TRIAL')
print('=' * 60)
print(f'  Trial #      : {best.number}')
print(f'  Val Loss     : {best.value:.6f}')
print(f'  Val Acc      : {best.user_attrs["val_acc"]:.6f}')
print()
print('  Best Params:')
for k, v in best.params.items():
    print(f'    {k:<20}: {v}')

# Statistik trial
pruned   = len([t for t in study.trials if t.state == TrialState.PRUNED])
complete = len([t for t in study.trials if t.state == TrialState.COMPLETE])
print(f'\n  Complete: {complete} | Pruned: {pruned}')

In [ ]:
# Top-10 trials berdasarkan val_loss
results = []
for t in study.trials:
    if t.state == TrialState.COMPLETE:
        row = {'trial': t.number, 'val_loss': round(t.value, 6),
               'val_acc': t.user_attrs.get('val_acc', None)}
        row.update(t.params)
        results.append(row)

df_res = pd.DataFrame(results).sort_values('val_loss')
print('Top-10 Trials:')
df_res.head(10)

## 9. Visualisasi


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Optimization history
ax1 = axes[0]
vals = [t.value for t in study.trials if t.state == TrialState.COMPLETE]
ax1.plot(range(len(vals)), vals, 'o-', color='steelblue', markersize=5)
ax1.axhline(min(vals), color='red', linestyle='--', label=f'Best: {min(vals):.4f}')
ax1.set_title('Optimization History (Val Loss)', fontsize=13)
ax1.set_xlabel('Completed Trial'); ax1.set_ylabel('Val Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Parameter importance (manual)
ax2 = axes[1]
importances = optuna.importance.get_param_importances(study)
params_imp = list(importances.keys())
imp_vals   = [importances[k] for k in params_imp]
ax2.barh(params_imp, imp_vals, color='coral')
ax2.set_title('Parameter Importance', fontsize=13)
ax2.set_xlabel('Importance Score')
ax2.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'optuna_result.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot tersimpan')

## 10. Simpan Best Params


In [ ]:
best_params = {
    'val_loss'       : round(best.value, 6),
    'val_acc'        : best.user_attrs['val_acc'],
    'hyperparameters': best.params
}

out_path = os.path.join(OUTPUT_DIR, 'optuna_best_params.json')
with open(out_path, 'w') as f:
    json.dump(best_params, f, indent=2)

print(f'✅ Best params disimpan ke: {out_path}')
print()
print('─' * 50)
for k, v in best.params.items():
    if isinstance(v, float):
        print(f'    {k:<20}: {v:.6f}')
    else:
        print(f'    {k:<20}: {v}')